In [2]:
%pip install pandas numpy matplotlib scikit-learn torch

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------

In [ ]:
# ============================================================
# IMDb 50K CUSTOMER REVIEW SENTIMENT ANALYZER
# LSTM + PyTorch
# ============================================================


# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

# Run this only if libraries are not installed
# %pip install pandas numpy matplotlib scikit-learn torch


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import re
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("Libraries imported successfully!")


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# 5. LOAD IMDb DATASET
# ============================================================

df = pd.read_csv("IMDB Dataset.csv")

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())

print("\nColumn Names:")
print(df.columns)

print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())


# ============================================================
# 6. CONVERT SENTIMENT TO NUMBERS
# ============================================================

# positive = 1
# negative = 0

df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print("\nNumeric Sentiment Distribution:")
print(df["sentiment"].value_counts())


# ============================================================
# 7. CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())


# Remove missing values

df = df.dropna().reset_index(drop=True)

print("\nDataset after removing missing values:")
print(df.shape)


# ============================================================
# 8. CLEAN TEXT
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(
        r"<br\s*/?>",
        " ",
        text
    )

    # Keep only letters and spaces
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["clean_review"] = df["review"].apply(
    clean_text
)


print("\nOriginal Review:")
print(df["review"].iloc[0])

print("\nCleaned Review:")
print(df["clean_review"].iloc[0])


# ============================================================
# 9. TOKENIZATION
# ============================================================

df["tokens"] = df["clean_review"].apply(
    lambda x: x.split()
)


print("\nTokenized Review:")
print(df["tokens"].iloc[0][:50])


# ============================================================
# 10. CREATE VOCABULARY
# ============================================================

from collections import Counter

word_counter = Counter()

for tokens in df["tokens"]:
    word_counter.update(tokens)


# Maximum vocabulary size

MAX_VOCAB_SIZE = 20000


# Special tokens

# PAD = 0
# UNK = 1

word_to_idx = {
    "<PAD>": 0,
    "<UNK>": 1
}


# Add most common words

most_common_words = word_counter.most_common(
    MAX_VOCAB_SIZE - 2
)


for idx, (word, count) in enumerate(
    most_common_words,
    start=2
):

    word_to_idx[word] = idx


print("\nVocabulary Size:")
print(len(word_to_idx))


# ============================================================
# 11. ENCODE WORDS
# ============================================================

def encode_review(tokens):

    return [
        word_to_idx.get(
            word,
            word_to_idx["<UNK>"]
        )
        for word in tokens
    ]


df["encoded"] = df["tokens"].apply(
    encode_review
)


print("\nOriginal Tokens:")
print(df["tokens"].iloc[0][:20])

print("\nEncoded Tokens:")
print(df["encoded"].iloc[0][:20])


# ============================================================
# 12. PAD SEQUENCES
# ============================================================

MAX_LENGTH = 200


def pad_sequence(
    sequence,
    max_length=MAX_LENGTH
):

    # Truncate long reviews

    if len(sequence) > max_length:

        return sequence[:max_length]


    # Pad short reviews

    return sequence + [
        0
    ] * (
        max_length - len(sequence)
    )


X = np.array(
    df["encoded"]
    .apply(pad_sequence)
    .tolist()
)

y = df["sentiment"].values


print("\nX Shape:")
print(X.shape)

print("\ny Shape:")
print(y.shape)


# ============================================================
# 13. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=SEED,

    stratify=y
)


print("\nTraining Data:")
print(X_train.shape)

print("\nTesting Data:")
print(X_test.shape)


# ============================================================
# 14. PYTORCH DATASET
# ============================================================

class IMDBDataset(Dataset):

    def __init__(
        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.long
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.X)


    def __getitem__(
        self,
        index
    ):

        return (
            self.X[index],
            self.y[index]
        )


# ============================================================
# 15. CREATE DATASETS
# ============================================================

train_dataset = IMDBDataset(
    X_train,
    y_train
)

test_dataset = IMDBDataset(
    X_test,
    y_test
)


# ============================================================
# 16. DATA LOADERS
# ============================================================

BATCH_SIZE = 64


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False
)


print("\nDataLoaders created successfully!")


# ============================================================
# 17. LSTM MODEL
# ============================================================

class SentimentLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3
    ):

        super(
            SentimentLSTM,
            self
        ).__init__()


        # ----------------------------------------------------
        # EMBEDDING
        # ----------------------------------------------------

        self.embedding = nn.Embedding(

            num_embeddings=vocab_size,

            embedding_dim=embedding_dim,

            padding_idx=0
        )


        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        self.lstm = nn.LSTM(

            input_size=embedding_dim,

            hidden_size=hidden_dim,

            num_layers=num_layers,

            batch_first=True,

            dropout=dropout
        )


        # ----------------------------------------------------
        # DROPOUT
        # ----------------------------------------------------

        self.dropout = nn.Dropout(
            dropout
        )


        # ----------------------------------------------------
        # FULLY CONNECTED
        # ----------------------------------------------------

        self.fc = nn.Linear(

            hidden_dim,

            1
        )


    def forward(
        self,
        x
    ):

        # ----------------------------------------------------
        # FIND REAL REVIEW LENGTH
        # ----------------------------------------------------

        lengths = (
            x != 0
        ).sum(
            dim=1
        ).cpu()


        # ----------------------------------------------------
        # EMBEDDING
        # ----------------------------------------------------

        embedded = self.embedding(x)


        # ----------------------------------------------------
        # PACK SEQUENCES
        # This makes the LSTM ignore padding.
        # ----------------------------------------------------

        packed = (
            nn.utils.rnn
            .pack_padded_sequence(
                embedded,

                lengths,

                batch_first=True,

                enforce_sorted=False
            )
        )


        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        packed_output, (
            hidden,
            cell
        ) = self.lstm(
            packed
        )


        # ----------------------------------------------------
        # LAST HIDDEN STATE
        # ----------------------------------------------------

        hidden = hidden[-1]


        # ----------------------------------------------------
        # DROPOUT
        # ----------------------------------------------------

        hidden = self.dropout(
            hidden
        )


        # ----------------------------------------------------
        # FULLY CONNECTED
        # ----------------------------------------------------

        output = self.fc(
            hidden
        )


        return output.squeeze(1)


# ============================================================
# 18. CREATE MODEL
# ============================================================

model = SentimentLSTM(

    vocab_size=len(word_to_idx),

    embedding_dim=128,

    hidden_dim=128,

    num_layers=2,

    dropout=0.3

).to(device)


print("\nModel Architecture:")
print(model)


# ============================================================
# 19. LOSS FUNCTION
# ============================================================

criterion = nn.BCEWithLogitsLoss()


# ============================================================
# 20. OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.0005
)


# ============================================================
# 21. TRAINING SETTINGS
# ============================================================

EPOCHS = 5


train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


# ============================================================
# 22. TRAIN MODEL
# ============================================================

print("\n")
print("=" * 60)
print("STARTING LSTM TRAINING")
print("=" * 60)


for epoch in range(EPOCHS):


    # ========================================================
    # TRAINING MODE
    # ========================================================

    model.train()


    total_train_loss = 0

    correct_train = 0

    total_train = 0


    for X_batch, y_batch in train_loader:


        # Move data to CPU/GPU

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)


        # Clear gradients

        optimizer.zero_grad()


        # Forward pass

        outputs = model(
            X_batch
        )


        # Calculate loss

        loss = criterion(

            outputs,

            y_batch
        )


        # Backpropagation

        loss.backward()


        # Prevent exploding gradients

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=5
        )


        # Update weights

        optimizer.step()


        # Add loss

        total_train_loss += (
            loss.item()
        )


        # Calculate predictions

        probabilities = torch.sigmoid(
            outputs
        )


        predictions = (
            probabilities >= 0.5
        ).float()


        # Count correct predictions

        correct_train += (

            predictions == y_batch

        ).sum().item()


        total_train += (
            y_batch.size(0)
        )


    # ========================================================
    # TRAINING RESULTS
    # ========================================================

    avg_train_loss = (

        total_train_loss /

        len(train_loader)
    )


    train_accuracy = (

        correct_train /

        total_train
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()


    total_val_loss = 0

    correct_val = 0

    total_val = 0


    with torch.no_grad():

        for X_batch, y_batch in test_loader:


            X_batch = X_batch.to(device)

            y_batch = y_batch.to(device)


            # Forward pass

            outputs = model(
                X_batch
            )


            # Validation loss

            loss = criterion(

                outputs,

                y_batch
            )


            total_val_loss += (
                loss.item()
            )


            # Probabilities

            probabilities = torch.sigmoid(
                outputs
            )


            # Predictions

            predictions = (

                probabilities >= 0.5

            ).float()


            # Correct

            correct_val += (

                predictions == y_batch

            ).sum().item()


            total_val += (
                y_batch.size(0)
            )


    # ========================================================
    # VALIDATION RESULTS
    # ========================================================

    avg_val_loss = (

        total_val_loss /

        len(test_loader)
    )


    val_accuracy = (

        correct_val /

        total_val
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    train_losses.append(
        avg_train_loss
    )

    val_losses.append(
        avg_val_loss
    )

    train_accuracies.append(
        train_accuracy
    )

    val_accuracies.append(
        val_accuracy
    )


    # ========================================================
    # DISPLAY
    # ========================================================

    print(

        f"Epoch [{epoch + 1}/{EPOCHS}] "

        f"| Train Loss: "
        f"{avg_train_loss:.4f} "

        f"| Val Loss: "
        f"{avg_val_loss:.4f} "

        f"| Train Acc: "
        f"{train_accuracy * 100:.2f}% "

        f"| Val Acc: "
        f"{val_accuracy * 100:.2f}%"

    )


print("=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)


# ============================================================
# 23. TRAINING & VALIDATION LOSS GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)


plt.plot(

    range(
        1,
        EPOCHS + 1
    ),

    train_losses,

    marker="o",

    label="Training Loss"
)


plt.plot(

    range(
        1,
        EPOCHS + 1
    ),

    val_losses,

    marker="o",

    label="Validation Loss"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Training and Validation Loss"
)

plt.xticks(
    range(
        1,
        EPOCHS + 1
    )
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 24. TRAINING & VALIDATION ACCURACY GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)


plt.plot(

    range(
        1,
        EPOCHS + 1
    ),

    train_accuracies,

    marker="o",

    label="Training Accuracy"
)


plt.plot(

    range(
        1,
        EPOCHS + 1
    ),

    val_accuracies,

    marker="o",

    label="Validation Accuracy"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Training and Validation Accuracy"
)

plt.xticks(
    range(
        1,
        EPOCHS + 1
    )
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 25. MODEL EVALUATION
# ============================================================

model.eval()


all_predictions = []

all_probabilities = []

all_labels = []


with torch.no_grad():

    for X_batch, y_batch in test_loader:


        X_batch = X_batch.to(device)


        # Model output

        outputs = model(
            X_batch
        )


        # Convert to probability

        probabilities = torch.sigmoid(
            outputs
        )


        # Convert probability to class

        predictions = (

            probabilities >= 0.5

        ).int()


        # Store

        all_predictions.extend(

            predictions
            .cpu()
            .numpy()
        )


        all_probabilities.extend(

            probabilities
            .cpu()
            .numpy()
        )


        all_labels.extend(

            y_batch.numpy()
        )


# ============================================================
# 26. CALCULATE METRICS
# ============================================================

accuracy = accuracy_score(

    all_labels,

    all_predictions
)


precision = precision_score(

    all_labels,

    all_predictions,

    zero_division=0
)


recall = recall_score(

    all_labels,

    all_predictions,

    zero_division=0
)


f1 = f1_score(

    all_labels,

    all_predictions,

    zero_division=0
)


# ============================================================
# 27. DISPLAY MODEL PERFORMANCE
# ============================================================

print("\n")
print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)


print(
    f"Accuracy  : "
    f"{accuracy * 100:.2f}%"
)


print(
    f"Precision : "
    f"{precision * 100:.2f}%"
)


print(
    f"Recall    : "
    f"{recall * 100:.2f}%"
)


print(
    f"F1-Score  : "
    f"{f1 * 100:.2f}%"
)


print("=" * 60)


# ============================================================
# 28. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:\n")


print(

    classification_report(

        all_labels,

        all_predictions,

        target_names=[
            "Negative",
            "Positive"
        ],

        zero_division=0
    )
)


# ============================================================
# 29. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    all_labels,

    all_predictions
)


print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(

    confusion_matrix=cm,

    display_labels=[
        "Negative",
        "Positive"
    ]
)


disp.plot()


plt.title(
    "Confusion Matrix"
)

plt.show()


# ============================================================
# 30. MODEL PERFORMANCE BAR CHART
# ============================================================

metric_names = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1-Score"

]


metric_values = [

    accuracy,

    precision,

    recall,

    f1

]


plt.figure(
    figsize=(10, 6)
)


bars = plt.bar(

    metric_names,

    metric_values
)


plt.ylim(
    0,
    1
)


plt.ylabel(
    "Score"
)


plt.title(
    "LSTM Model Performance"
)


# Add percentage labels

for bar, value in zip(

    bars,

    metric_values

):

    plt.text(

        bar.get_x()
        + bar.get_width() / 2,

        value + 0.02,

        f"{value * 100:.2f}%",

        ha="center"

    )


plt.grid(

    axis="y",

    alpha=0.3
)


plt.show()


# ============================================================
# 31. SENTIMENT PREDICTION FUNCTION
# ============================================================

def predict_sentiment(review):


    # Set model to evaluation mode

    model.eval()


    # --------------------------------------------------------
    # CLEAN
    # --------------------------------------------------------

    cleaned_review = clean_text(
        review
    )


    # --------------------------------------------------------
    # TOKENIZE
    # --------------------------------------------------------

    tokens = cleaned_review.split()


    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    encoded = [

        word_to_idx.get(

            word,

            word_to_idx["<UNK>"]

        )

        for word in tokens

    ]


    # --------------------------------------------------------
    # PAD
    # --------------------------------------------------------

    padded = pad_sequence(
        encoded
    )


    # --------------------------------------------------------
    # CONVERT TO TENSOR
    # --------------------------------------------------------

    input_tensor = torch.tensor(

        [padded],

        dtype=torch.long

    ).to(device)


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    with torch.no_grad():

        output = model(
            input_tensor
        )


        probability = torch.sigmoid(
            output
        ).item()


    # --------------------------------------------------------
    # SENTIMENT
    # --------------------------------------------------------

    if probability >= 0.5:

        sentiment = "Positive"

        confidence = probability

    else:

        sentiment = "Negative"

        confidence = 1 - probability


    return sentiment, confidence


# ============================================================
# 32. TEST YOUR REQUIRED INPUT
# ============================================================

review = (
    "The movie was excellent and very enjoyable."
)


sentiment, confidence = predict_sentiment(
    review
)


print("\n")
print("=" * 60)
print("SENTIMENT ANALYZER")
print("=" * 60)


print(
    "\nInput Review:"
)

print(review)


print(
    "\nPredicted Sentiment:"
)

print(sentiment)


print(
    f"\nConfidence: "
    f"{confidence * 100:.2f}%"
)


print("=" * 60)


# ============================================================
# 33. TEST MULTIPLE REVIEWS
# ============================================================

test_reviews = [

    "The movie was excellent and very enjoyable.",

    "This is one of the best movies I have ever watched.",

    "The movie was terrible and extremely boring.",

    "I hated this movie. It was a complete waste of time.",

    "Amazing acting and wonderful story.",

    "The film was disappointing and poorly made."

]


print("\n")
print("=" * 60)
print("MULTIPLE REVIEW TESTING")
print("=" * 60)


for review in test_reviews:


    sentiment, confidence = (
        predict_sentiment(review)
    )


    print("\nReview:")
    print(review)


    print(
        "Sentiment:",
        sentiment
    )


    print(
        f"Confidence: "
        f"{confidence * 100:.2f}%"
    )


# ============================================================
# 34. SAVE MODEL
# ============================================================

torch.save(

    model.state_dict(),

    "imdb_lstm_sentiment_model.pth"

)


print(
    "\nModel saved successfully:"
)

print(
    "imdb_lstm_sentiment_model.pth"
)


# ============================================================
# 35. SAVE VOCABULARY
# ============================================================

with open(

    "word_to_idx.pkl",

    "wb"

) as file:

    pickle.dump(

        word_to_idx,

        file
    )


print(
    "Vocabulary saved successfully:"
)

print(
    "word_to_idx.pkl"
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("FINAL PROJECT RESULT")
print("=" * 60)


print(
    f"Accuracy  : "
    f"{accuracy * 100:.2f}%"
)


print(
    f"Precision : "
    f"{precision * 100:.2f}%"
)


print(
    f"Recall    : "
    f"{recall * 100:.2f}%"
)


print(
    f"F1-Score  : "
    f"{f1 * 100:.2f}%"
)


print("\nExample Input:")
print(
    "The movie was excellent and very enjoyable."
)


example_sentiment, example_confidence = (
    predict_sentiment(
        "The movie was excellent and very enjoyable."
    )
)


print(
    "\nPredicted Sentiment:",
    example_sentiment
)


print(
    f"Confidence: "
    f"{example_confidence * 100:.2f}%"
)


print("=" * 60)
print("PROJECT COMPLETED")
print("=" * 60)

Libraries imported successfully!
Using device: cpu

Dataset Shape:
(50000, 2)

First 5 Rows:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Column Names:
Index(['review', 'sentiment'], dtype='str')

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Numeric Sentiment Distribution:
sentiment
1    25000
0    25000
Name: count, dtype: int64

Missing Values:
review       0
sentiment    0
dtype: int64

Dataset after removing missing values:
(50000, 2)

Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me